In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

###############################################################################
# 1) Assign each tokenizer family and define a "darker" colormap for each
###############################################################################
def identify_method(filename):
    """Return the tokenizer family based on filename."""
    f = filename.lower()
    if "directsam" in f or "epoc" in f:
        return "directsam"
    elif "panoptic" in f:
        return "panoptic"
    elif "sam" in f:
        # Make sure "sam" is not confused with "directsam" above
        return "sam"
    elif "superpixel" in f or "slic" in f:
        return "superpixel"
    else:
        return "others"  # fallback

def get_darker_cmap(base_cmap, min_val=0.5, max_val=1.0):
    """
    Return a truncated version of base_cmap limited to [min_val, max_val] 
    so the colors are darker. 
    """
    new_cmap = mpl.colors.LinearSegmentedColormap.from_list(
        "trunc({n}, {a:.2f}~{b:.2f})".format(n=base_cmap.name, a=min_val, b=max_val),
        base_cmap(np.linspace(min_val, max_val, 256))
    )
    return new_cmap

FAMILY_COLORMAP = {
    "directsam":  get_darker_cmap(plt.cm.Blues),
    "panoptic":   get_darker_cmap(plt.cm.Greens),
    "sam":        get_darker_cmap(plt.cm.Oranges),
    "superpixel": get_darker_cmap(plt.cm.Purples),
    "others":     get_darker_cmap(plt.cm.Greys),
}

###############################################################################
# 2) Load all your JSON logs and group them by family
###############################################################################
root = "outputs/efficiency_test/V100-30cpu"
json_files = sorted([f for f in os.listdir(root) if f.endswith(".json")])

families_data = {
    "directsam":  [],
    "panoptic":   [],
    "sam":        [],
    "superpixel": [],
    "others":     [],
}

all_n_proc_vals = []

for fname in json_files:
    filepath = os.path.join(root, fname)
    with open(filepath, "r") as f:
        data = json.load(f)

    config_file = data["args"]["tokenizer_config"].split("/")[-1]
    family      = identify_method(config_file)
    families_data[family].append(data)

    # Collect n_proc for global color normalization
    for exp in data["data"].values():
        all_n_proc_vals.append(exp["n_proc"])

# Prepare normalization for color
n_min, n_max = min(all_n_proc_vals), max(all_n_proc_vals)
norm = plt.Normalize(vmin=n_min, vmax=n_max)

###############################################################################
# 3) Helper for computing average GPU usage across all processes
###############################################################################
def avg_gpu_utilization(per_proc):
    utils = [p["avg_gpu_util_percent"] for p in per_proc.values()]
    return float(np.mean(utils)) if utils else 0.0

###############################################################################
# 4) Families of interest in a defined order
###############################################################################
families_of_interest = ["panoptic", "superpixel",  "sam", "directsam", ]

###############################################################################
# 5) Two figures:
#    - fig_fps -> subplots for "FPS vs. #proc"
#    - fig_gpu -> subplots for "GPU usage vs. #proc"
###############################################################################
fig_fps, axs_fps = plt.subplots(1, len(families_of_interest), 
                                figsize=(25, 5), sharey=True)
fig_gpu, axs_gpu = plt.subplots(1, len(families_of_interest), 
                                figsize=(25, 5), sharey=True)

# We'll remove figure-wide suptitle or subplot titles:
#   (User specifically requested "No title and suptitle".)

# Minimal axis labeling for clarity
axs_fps[0].set_ylabel("Parallel Throughput (FPS)", fontsize=11)
axs_gpu[0].set_ylabel("Avg GPU Util (%)", fontsize=11)

for i, family in enumerate(families_of_interest):
    ax_f = axs_fps[i]
    ax_g = axs_gpu[i]

    # x-axis label
    ax_f.set_xlabel("Number of Processes", fontsize=11)
    ax_f.grid(True)
    ax_g.set_xlabel("Number of Processes", fontsize=11)
    ax_g.grid(True)

    # Grab the chosen colormap for this family
    cmap = FAMILY_COLORMAP[family]

    # If no data, skip
    if len(families_data[family]) == 0:
        continue

    # Plot each configuration within this family
    for data in families_data[family]:
        config_file = data["args"]["tokenizer_config"].split("/")[-1]
        label = f"{config_file}_max{data['args'].get('max_tokens', 'N/A')}"

        # Collect data across different n_proc
        valid_points = []
        oom_points   = []  # store (n_proc, fps) for OOM runs
        gpu_points   = []

        for exp in data["data"].values():
            n_proc   = exp["n_proc"]
            fps_val  = exp["parallel_throughput"]
            is_oom   = exp["oom_happened"]
            gpu_val  = avg_gpu_utilization(exp["per_proc"])

            if is_oom:
                oom_points.append((n_proc, fps_val, gpu_val))
            else:
                valid_points.append((n_proc, fps_val, gpu_val))

        # Sort by n_proc
        valid_points.sort(key=lambda x: x[0])
        oom_points.sort(key=lambda x: x[0])

        # (a) Plot FPS
        for (n_proc, fps_val, _) in valid_points:
            color = cmap(norm(n_proc))
            ax_f.scatter(n_proc, fps_val, color=color, s=10)
            ax_f.scatter(n_proc, fps_val, color=color, s=60, alpha=0.3)

        for (n_proc, fps_val, _) in oom_points:
            # Mark red 'x'
            ax_f.scatter(n_proc, fps_val, marker='x', color='red', s=40)

        # Connect valid points with a line (dark color from the last n_proc, skipping OOM)
        if len(valid_points) > 1:
            # color of the largest n_proc
            final_color = cmap(norm(valid_points[-1][0]))
            xs = [p[0] for p in valid_points]
            ys = [p[1] for p in valid_points]
            ax_f.plot(xs, ys, color=final_color, label=label)

        # (b) Plot GPU usage
        for (n_proc, _, gpu_val) in valid_points:
            color = cmap(norm(n_proc))
            ax_g.scatter(n_proc, gpu_val, color=color, s=10)
            ax_g.scatter(n_proc, gpu_val, color=color, s=60, alpha=0.3)

        for (n_proc, _, gpu_val) in oom_points:
            ax_g.scatter(n_proc, gpu_val, marker='x', color='red', s=40)

        if len(valid_points) > 1:
            final_color = cmap(norm(valid_points[-1][0]))
            xs = [p[0] for p in valid_points]
            ys = [p[2] for p in valid_points]
            ax_g.plot(xs, ys, color=final_color, label=label)

    ax_f.legend(fontsize=8)
    ax_g.legend(fontsize=8)

    # plt.tight_layout()
    # tight layout for both:
    fig_fps.subplots_adjust(wspace=0.1)
    fig_gpu.subplots_adjust(wspace=0.1)

    # xlim[0, 30] for both
    ax_f.set_xlim(0, 31)
    ax_g.set_xlim(0, 31)

plt.show()

###############################################################################
# 6) Optional: Summarize best performance in a CSV
###############################################################################
summary = {}
for family, data_list in families_data.items():
    for data in data_list:
        config_file = data["args"]["tokenizer_config"].split("/")[-1]
        label = f"{config_file}_max{data['args'].get('max_tokens', 'N/A')}"

        best_fps   = 0.0
        best_nproc = 0
        best_gpu   = 0.0

        for exp in data["data"].values():
            if exp["oom_happened"]:
                continue
            thrpt = exp["parallel_throughput"]
            gpu_u = avg_gpu_utilization(exp["per_proc"])
            n_proc = exp["n_proc"]
            if thrpt > best_fps:
                best_fps = thrpt
                best_nproc = n_proc
                best_gpu   = gpu_u

        summary[label] = {
            "family":   family,
            "best_fps": round(best_fps, 2),
            "best_nproc": best_nproc,
            "gpu_usage_at_best": round(best_gpu, 2),
        }

df = pd.DataFrame(summary).T
df.to_csv(os.path.join(root, "summary.csv"))
print(df)

In [ ]:
import os, json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


root = 'outputs/efficiency_test/V100-30cpu'


results = []
files = sorted(os.listdir(root))
for file in files:
    if file.endswith('.json'):
        with open(os.path.join(root, file), 'r') as f:
            results.append(json.load(f))

print('Results:', len(results))

def calculate_gpu_ultilization(per_proc):
    # print(per_proc)
    per_proc_gpu_ultilization = []
    for proc in per_proc.values():
        per_proc_gpu_ultilization.append(proc['avg_gpu_util_percent'])
    return np.mean(per_proc_gpu_ultilization)


summary = {}
plt.figure(figsize=(20, 20))
for result in results:
    tokenizer = result['args']['tokenizer_config'].split('/')[-1] + f"_{result['args']['max_tokens']}"
    fps = []
    gpu_ultilization = []
    n_processes = []

    # print(result['args'])
    for experiment in result['data'].values():
        if experiment['oom_happened']:
            plt.scatter(experiment['n_proc'], experiment['parallel_throughput'], color='red', marker='x')

            continue
        fps.append(experiment['parallel_throughput'])
        gpu_ultilization.append(calculate_gpu_ultilization(experiment['per_proc']))
        n_processes.append(experiment['n_proc'])

    plt.plot(n_processes, fps, label=tokenizer)
    plt.scatter(n_processes, fps)
    plt.text(n_processes[-1], fps[-1], tokenizer, fontsize=9)

    summary[tokenizer] = {
        'Max FPS':  max(fps),
        'Max processes': n_processes[fps.index(max(fps))],
        'Max GPU Utilization': gpu_ultilization[fps.index(max(fps))]
    }

# df = pd.DataFrame(max_fps.items(), columns=['Tokenizer', 'Max FPS'])
df = pd.DataFrame(summary).T
df.to_csv('outputs/efficiency_test/summary.csv')
print(df)


plt.legend()
plt.xlabel('Number of processes')
plt.ylabel('FPS')

# plt.ylim(0, 25)

# add grid
plt.grid()
plt.show()



In [ ]:

import os
import json
import matplotlib.pyplot as plt
import numpy as np

# A simple color mapping for different methods
METHOD_COLORS = {
    "panoptic": "green",
    "SAM": "orange",
    "SLIC": "purple",
    "EPOC": "blue"
}

def identify_method(filename):
    filename_lower = filename.lower()
    if "panoptic" in filename_lower:
        return "panoptic"
    elif "sam" in filename_lower:
        return "SAM"
    elif "slic" in filename_lower or "superpixel" in filename_lower:
        return "SLIC"
    elif "directsam" in filename_lower or "epoc" in filename_lower:
        return "EPOC"
    else:
        return "unknown"

# Directory with JSON logs
LOG_DIR = "outputs/efficiency_test/V100-30cpu"
json_files = [f for f in os.listdir(LOG_DIR) if f.endswith(".json")]

plt.figure(figsize=(8, 6))

for file in sorted(json_files):
    print(file)
    method = identify_method(file)
    color = METHOD_COLORS.get(method, "gray")
    
    with open(os.path.join(LOG_DIR, file), "r") as f:
        data = json.load(f)
    
    # The "data" field is a dict of experiments keyed by process count
    experiments = data["data"].values()
    n_procs_list = []
    thrpt_list   = []
    
    for exp in experiments:
        n_proc = exp["n_proc"]
        if exp["oom_happened"]:
            # Plot an 'X' at y=0 to indicate OOM
            plt.scatter(n_proc, thrpt, color="red", marker="x", zorder=3)
            continue
        # Parallel throughput
        thrpt = calculate_gpu_ultilization(experiment['per_proc'])
        # thrpt = exp["parallel_throughput"]
        n_procs_list.append(n_proc)
        thrpt_list.append(thrpt)
    
    # Plot each tokenizer as a line
    if len(n_procs_list) > 0:
        plt.plot(n_procs_list, thrpt_list, label=file, color=color, linewidth=2)
        plt.scatter(n_procs_list, thrpt_list, color=color, s=40)
    
plt.xlabel("Number of Processes", fontsize=12)
plt.ylabel("Parallel Throughput (images/sec)", fontsize=12)
plt.title("Tokenization Efficiency Scaling", fontsize=14)
plt.legend(fontsize=8, loc="upper left", bbox_to_anchor=(1,1))
plt.grid(True)

# Save as PDF (for Overleaf)
os.makedirs("figs", exist_ok=True)
plt.savefig("figs/efficiency_scaling.pdf", bbox_inches="tight")
plt.show()